# RNN 实现古诗生成 —— 七言绝句

以 **"明月"** 为起始词，基于双层 LSTM 生成七言绝句。

**环境要求**
```bash
pip install torch matplotlib
```

**使用方法**：将本 Notebook 与四个 JSON 数据文件放在同一目录，逐单元格运行即可。

## 0. 安装依赖（首次运行时取消注释）

In [1]:
# !pip install torch matplotlib

## 1. 导入库 & 全局配置

In [2]:
import json, os, re, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ─── 全局配置（按需修改） ───────────────────────────────────────────
CONFIG = {
    # ── 数据文件 ──────────────────────────────────────
    # num_data_files: 读取前 N 个 JSON 文件（按文件名排序后取前 N 个）
    # 设为 -1 表示读取 data_dir 下所有 poet.song.*.json 文件
    "num_data_files": 80,
    "data_dir": "data_files",

    # 模型超参
    "embedding_dim": 512,
    "hidden_size":   1024,
    "num_layers":    3,
    "dropout":       0.3,   # 原 0.1 → 0.3，降低 33M 参数模型的过拟合风险

    # 训练超参
    "batch_size":    128,
    "num_epochs":    60,
    "learning_rate": 1e-3,
    "lr_decay_step": 15,    # 原 10 → 15，每 15 轮衰减一次，避免后期 LR 过低
    "lr_decay_gamma":0.7,   # 原 0.5 → 0.7，衰减更温和（60轮后约 1e-3×0.7³≈3.4e-4）
    "clip_grad":     5.0,
    "seed":          42,

    # 生成参数
    "start_words":   "明月",
    "temperature":   0.8,    # 采样温度，越低生成越保守

    # 输出路径
    "save_model":    "poem_lstm.pth",
    "loss_fig":      "training_loss.png",
}

# ─── 随机种子 & 设备 ─────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 特殊 token
PAD, START, END, UNK = "<PAD>", "<START>", "<END>", "<UNK>"

Using device: cuda


## 2. 数据加载与预处理

**数据集格式说明**：JSON 中七言绝句存储为 **2 个 paragraph**，每个 16 字：
```
"XXXXXXX，XXXXXXX。"   ← 上下两句合为一联
```
过滤后展开为长度 32 的字符序列。

In [3]:
# 七言绝句每联格式：7汉字 + 逗号 + 7汉字 + 句号/！/？
_QIYAN_PATTERN = re.compile(
    r'^[一-鿿]{7}[，,][一-鿿]{7}[。！？]$'
)

def is_qiyan_jueju(paragraphs):
    """判断是否为七言绝句（2联，每联16字）"""
    if len(paragraphs) != 2:
        return False
    for p in paragraphs:
        ps = p.strip()
        if len(ps) != 16 or not _QIYAN_PATTERN.match(ps):
            return False
    return True


def load_sequences(data_dir, filenames):
    """读取 JSON，过滤七言绝句，每首展开为 32 字符序列（含标点）"""
    sequences = []
    for fname in filenames:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"[warning] 文件不存在，跳过: {fpath}")
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            para = item.get("paragraphs", [])
            if is_qiyan_jueju(para):
                seq = "".join(p.strip() for p in para)  # 长度 32
                sequences.append(seq)

    print(f"过滤后七言绝句: {len(sequences)} 首")
    assert len(sequences) > 0, "未找到七言绝句！请检查 data_dir 配置。"
    return sequences


def build_vocab(sequences):
    """构建字符级词表"""
    chars = sorted(set("".join(sequences)))
    vocab = [PAD, START, END, UNK] + chars
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"词表大小: {len(vocab)}")
    return char2idx, idx2char, vocab


# ── 自动发现数据文件（按文件名排序） ────────────────────────────────
all_json = sorted([
    f for f in os.listdir(CONFIG["data_dir"])
    if f.startswith("poet.song") and f.endswith(".json")
])
n = CONFIG["num_data_files"]
data_files = all_json[:n] if n > 0 else all_json
print(f"共发现 {len(all_json)} 个 JSON 文件，将读取其中 {len(data_files)} 个")

sequences = load_sequences(CONFIG["data_dir"], data_files)
char2idx, idx2char, vocab = build_vocab(sequences)

# 查看几条示例
print("\n示例序列：")
for s in sequences[:3]:
    print(" ", s)

共发现 255 个 JSON 文件，将读取其中 80 个
过滤后七言绝句: 23213 首
词表大小: 6564

示例序列：
  欲出未出光辣達，千山萬山如火發。須臾走向天上來，逐却殘星趕却月。
  片片飛來靜又閑，樓頭江上復山前。飄零盡日不歸去，帖破清光萬里天。
  一氣東南王斗牛，祖龍潜爲子孫憂。金陵地脈何曾斷，不覺真人已姓劉。


## 3. 构建 PyTorch Dataset

In [4]:
class PoemDataset(Dataset):
    """
    每个样本 (inp, tgt)，序列长度均为 32：
        inp = [START] + seq[:-1]   （前一字，i=0 时为 START）
        tgt = seq                   （当前字，作为预测目标）
    Teacher Forcing 训练范式。
    """
    def __init__(self, sequences, char2idx):
        start_id = char2idx[START]
        unk_id   = char2idx[UNK]
        self.data = []
        for seq in sequences:
            ids = [char2idx.get(c, unk_id) for c in seq]
            inp = torch.tensor([start_id] + ids[:-1], dtype=torch.long)
            tgt = torch.tensor(ids, dtype=torch.long)
            self.data.append((inp, tgt))

    def __len__(self):            return len(self.data)
    def __getitem__(self, i):     return self.data[i]


dataset = PoemDataset(sequences, char2idx)
print(f"训练样本数: {len(dataset)}")
inp_sample, tgt_sample = dataset[0]
print(f"inp shape: {inp_sample.shape},  tgt shape: {tgt_sample.shape}")


训练样本数: 23213
inp shape: torch.Size([32]),  tgt shape: torch.Size([32])


## 4. 模型定义

结构：**Embedding(512) → 3层 LSTM(1024) → Dropout(0.3) → Projection(512) → Linear(vocab_size)**

**权重共享（Weight Tying）**：输出线性层 `fc` 与 `Embedding` 矩阵共享权重，减少约 340 万参数，同时缓解过拟合。

In [5]:
class PoemLSTM(nn.Module):
    """
    字符级 LSTM 语言模型
    Embedding(512) → 3层LSTM(1024) → Dropout(0.3) → Projection(512) → Linear(vocab_size)

    Weight Tying：fc.weight 与 embedding.weight 共享，减少约 340 万参数。
    因 hidden_size(1024) ≠ emb_dim(512)，通过 proj 层对齐维度后再做共享。
    """
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = emb_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        # 将 LSTM 输出从 hidden_size 投影回 emb_dim，以便与 Embedding 权重共享
        self.proj = nn.Linear(hidden_size, emb_dim, bias=False)
        # bias=False：权重与 embedding 共享，bias 不参与共享
        self.fc   = nn.Linear(emb_dim, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight   # ← Weight Tying

        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, x, hidden=None):
        """x: (B, L)  →  logits: (B, L, V), hidden"""
        emb    = self.dropout(self.embedding(x))        # (B, L, E)
        out, hidden = self.lstm(emb, hidden)             # (B, L, H)
        out    = self.proj(self.dropout(out))            # (B, L, E)  投影
        logits = self.fc(out)                            # (B, L, V)  共享权重
        return logits, hidden

    def init_hidden(self, batch, device):
        h = torch.zeros(self.lstm.num_layers, batch,
                        self.lstm.hidden_size, device=device)
        return (h, torch.zeros_like(h))


# 实例化模型
model = PoemLSTM(
    vocab_size  = len(vocab),
    emb_dim     = CONFIG["embedding_dim"],
    hidden_size = CONFIG["hidden_size"],
    num_layers  = CONFIG["num_layers"],
    dropout     = CONFIG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {n_params:,}  （权重共享后，较原版减少约 337 万参数）")
print(model)

模型参数量: 26,978,304  （权重共享后，较原版减少约 337 万参数）
PoemLSTM(
  (embedding): Embedding(6564, 512, padding_idx=0)
  (lstm): LSTM(512, 1024, num_layers=3, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (proj): Linear(in_features=1024, out_features=512, bias=False)
  (fc): Linear(in_features=512, out_features=6564, bias=False)
)


## 5. 训练

In [6]:
import math

def train_epoch(model, loader, optimizer, criterion, device, clip):
    """训练一个 epoch，返回平均 per-token loss"""
    model.train()
    total_loss, total_n = 0.0, 0
    for inp, tgt in loader:
        inp, tgt = inp.to(device), tgt.to(device)
        hidden = model.init_hidden(inp.size(0), device)

        logits, _ = model(inp, hidden)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item() * tgt.numel()
        total_n    += tgt.numel()
    return total_loss / total_n


# ─── 初始化训练组件 ─────────────────────────────────────────────────
loader    = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                       shuffle=True, num_workers=0)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=CONFIG["lr_decay_step"],
    gamma=CONFIG["lr_decay_gamma"])
criterion = nn.CrossEntropyLoss()

# ─── 训练循环 ───────────────────────────────────────────────────────
epoch_losses = []

for epoch in range(1, CONFIG["num_epochs"] + 1):
    loss = train_epoch(model, loader, optimizer, criterion,
                       DEVICE, CONFIG["clip_grad"])
    scheduler.step()
    epoch_losses.append(loss)

    lr_now = optimizer.param_groups[0]["lr"]

    if epoch % 5 == 0:
        ppl = math.exp(loss)
        print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
              f"Loss: {loss:.4f}  PPL: {ppl:7.2f}  LR: {lr_now:.2e}  ← Perplexity")
    else:
        print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
              f"Loss: {loss:.4f}  LR: {lr_now:.2e}")

print("\n训练完成！")
print(f"最终 Loss: {epoch_losses[-1]:.4f}  "
      f"最终 PPL: {math.exp(epoch_losses[-1]):.2f}")

Epoch [01/60]  Loss: 6.5692  LR: 1.00e-03
Epoch [02/60]  Loss: 5.6843  LR: 1.00e-03
Epoch [03/60]  Loss: 5.3298  LR: 1.00e-03
Epoch [04/60]  Loss: 5.1311  LR: 1.00e-03
Epoch [05/60]  Loss: 4.9857  PPL:  146.31  LR: 1.00e-03  ← Perplexity
Epoch [06/60]  Loss: 4.8658  LR: 1.00e-03
Epoch [07/60]  Loss: 4.7607  LR: 1.00e-03
Epoch [08/60]  Loss: 4.6686  LR: 1.00e-03
Epoch [09/60]  Loss: 4.5833  LR: 1.00e-03
Epoch [10/60]  Loss: 4.5025  PPL:   90.25  LR: 1.00e-03  ← Perplexity
Epoch [11/60]  Loss: 4.4279  LR: 1.00e-03
Epoch [12/60]  Loss: 4.3581  LR: 1.00e-03
Epoch [13/60]  Loss: 4.2910  LR: 1.00e-03
Epoch [14/60]  Loss: 4.2283  LR: 1.00e-03
Epoch [15/60]  Loss: 4.1677  PPL:   64.57  LR: 7.00e-04  ← Perplexity
Epoch [16/60]  Loss: 4.0514  LR: 7.00e-04
Epoch [17/60]  Loss: 3.9814  LR: 7.00e-04
Epoch [18/60]  Loss: 3.9263  LR: 7.00e-04
Epoch [19/60]  Loss: 3.8761  LR: 7.00e-04
Epoch [20/60]  Loss: 3.8298  PPL:   46.06  LR: 7.00e-04  ← Perplexity
Epoch [21/60]  Loss: 3.7862  LR: 7.00e-04
Epoch 

## 6. 保存模型

In [7]:
torch.save({
    "model_state_dict": model.state_dict(),
    "char2idx": char2idx,
    "idx2char":  idx2char,
    "vocab":     vocab,
    "config":    CONFIG,
}, CONFIG["save_model"])
print(f"模型已保存 → {CONFIG['save_model']}")


模型已保存 → poem_lstm.pth


## 7. 绘制 Loss 收敛曲线

In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
epochs_x = list(range(1, len(epoch_losses) + 1))
ax.plot(epochs_x, epoch_losses, "g-o", markersize=2,
        linewidth=1.8, label="Train Loss")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Loss",  fontsize=13)
ax.set_title("Training Loss Curve", fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(epochs_x)
fig.tight_layout()
fig.savefig(CONFIG["loss_fig"], dpi=150)
plt.show()
print(f"Loss 曲线 → {CONFIG['loss_fig']}")


Loss 曲线 → training_loss.png


C:\Windows\Temp\ipykernel_31920\1642332493.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 8. 定义生成函数

**生成策略**：
1. 用 `[START] + start_words` 预热 LSTM hidden state
2. 从 `start_words` 最后一字开始自回归采样
3. 在标点位置（索引 7 / 15 / 23 / 31）**强制**插入正确标点
4. 其余位置屏蔽所有标点 token，再按温度 softmax 采样

In [9]:
# 七言绝句标点强制位置（0-based 序列索引）
PUNCT_MAP   = {7: "，", 15: "。", 23: "，", 31: "。"}
ALL_PUNCTS  = set("，。！？；、,.")


def generate(model, start_words, char2idx, idx2char, device,
             temperature=1.0, seq_len=32):
    """
    以 start_words 为前缀，自回归生成一首七言绝句（32 字含标点）。
    返回格式化的 4 行字符串。
    """
    model.eval()
    unk_id   = char2idx[UNK]
    start_id = char2idx[START]
    all_punct_ids = [char2idx[p] for p in ALL_PUNCTS if p in char2idx]

    generated = list(start_words)

    with torch.no_grad():
        # 预热 hidden state
        primer = [start_id] + [char2idx.get(c, unk_id) for c in start_words]
        inp    = torch.tensor([primer], dtype=torch.long, device=device)
        hidden = model.init_hidden(1, device)
        _, hidden = model(inp, hidden)

        # 自回归生成
        last_id = char2idx.get(start_words[-1], unk_id)
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)

        while len(generated) < seq_len:
            logits, hidden = model(inp, hidden)
            logit = logits[0, 0].clone() / temperature

            pos = len(generated)

            if pos in PUNCT_MAP:
                next_char = PUNCT_MAP[pos]
            else:
                logit[all_punct_ids] = -1e9
                probs = torch.softmax(logit, dim=-1)
                next_id = torch.multinomial(probs, 1).item()
                next_char = idx2char.get(next_id, UNK)

            generated.append(next_char)
            inp = torch.tensor(
                [[char2idx.get(next_char, unk_id)]],
                dtype=torch.long, device=device
            )

    s     = "".join(generated[:seq_len])
    lines = [s[i*8:(i+1)*8] for i in range(4)]
    return "\n".join(lines)

print("生成函数定义完毕。")


生成函数定义完毕。


## 9. 生成古诗

In [10]:
print("=" * 50)
print(f'以「{CONFIG["start_words"]}」为起始词生成七言绝句：')
print("=" * 50)

for i in range(5):
    poem = generate(
        model, CONFIG["start_words"],
        char2idx, idx2char, DEVICE,
        temperature=CONFIG["temperature"]
    )
    print(f"\n【第 {i+1} 首】\n{poem}")

print("\n" + "=" * 50)


以「明月」为起始词生成七言绝句：

【第 1 首】
明月明波影裏遊，
此涼應不爲清秋。
斜陽作陣湖山雨，
洗盡塵埃數日愁。

【第 2 首】
明月明如火炬天，
此時此夜轉孤燈。
兵邊物物自無寐，
獨倚蒲團夜夢中。

【第 3 首】
明月塘風雨夜凉，
夢魂猶在鐵冠前。
龍深日照長安近，
山上龍門一榻清。

【第 4 首】
明月明溪氣清輝，
棹來風露不勝涼。
如何一夜蟾松露，
未減人間萬木花。

【第 5 首】
明月明行照初干，
此月今宵月在天。
夜靜釣魚天籟息，
雲頭頻見一雙橫。



## 10. （可选）加载已保存模型重新生成

如果已经训练并保存了模型，可以跳过训练步骤，直接从这里加载生成。

In [11]:
# checkpoint = torch.load(CONFIG["save_model"], map_location=DEVICE)
# 
# char2idx = checkpoint["char2idx"]
# idx2char = checkpoint["idx2char"]
# vocab    = checkpoint["vocab"]
# 
# model_loaded = PoemLSTM(
#     vocab_size  = len(vocab),
#     emb_dim     = CONFIG["embedding_dim"],
#     hidden_size = CONFIG["hidden_size"],
#     num_layers  = CONFIG["num_layers"],
#     dropout     = CONFIG["dropout"],
# ).to(DEVICE)
# model_loaded.load_state_dict(checkpoint["model_state_dict"])
# 
# poem = generate(model_loaded, "明月", char2idx, idx2char, DEVICE)
# print(poem)
